# Titanic Machine Learning Project\n\nThis notebook demonstrates the end-to-end machine learning pipeline for the Titanic survival prediction challenge. It is designed to run seamlessly on Google Colab, Kaggle kernels, or a local environment by importing modular code from the `src/` directory.

## Setup and Environment Detection\nDetect the current runtime environment (Kaggle, Colab, or Local) and configure paths/dependencies accordingly.

In [ ]:
# Detect environment and setup
import os, sys

# Environment detection
IS_KAGGLE = os.path.exists('/kaggle/input')
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
IS_LOCAL = not IS_KAGGLE and not IS_COLAB

print(f'Environment: {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "Local"}')

# Setup for Colab: clone repo and install deps
if IS_COLAB:
    !git clone https://github.com/10Unknownboy/Titanic-ML-Model.git /content/Titanic-ML-Model 2>/dev/null || true
    %cd /content/Titanic-ML-Model
    !pip install -q -r requirements.txt

# Setup for Kaggle: install deps and add src to path
if IS_KAGGLE:
    # If src/ was uploaded as a dataset or utility script
    kaggle_src = '/kaggle/input/titanic-ml-src'
    if os.path.exists(kaggle_src):
        sys.path.insert(0, kaggle_src)
    # Also try working directory
    sys.path.insert(0, '/kaggle/working')

# For all environments, ensure project root is on path
if IS_LOCAL:
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    sys.path.insert(0, project_root)

## Imports\nImporting standard libraries and project modules.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

# Import from src modules
from src.config import Config
from src.data_loader import load_data
from src.feature_engineering import TitanicFeatureEngineer
from src.preprocessing import get_preprocessor
from src.models import get_base_models, StackingEnsemble
from src.training import train_pipeline
from src.evaluation import evaluate_individual_models, print_evaluation_report
from src.submission import generate_submission, validate_submission
from src.utils import setup_logging, set_seed

## Configuration and Initialization\nInitialize the project configuration, setup the logging mechanism, and set random seeds for reproducibility.

In [ ]:
# Initialize Config and Setup
config = Config()
logger = setup_logging()
set_seed(config.RANDOM_STATE)
logger.info("Project initialized with configuration.")

## Data Loading\nLoad the training and testing datasets and perform a basic inspection of shapes, data types, and missing values.

In [ ]:
train_df, test_df = load_data(config)

print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)

print("\n--- First 5 rows of Training Data ---")
display(train_df.head())

print("\n--- Info ---")
display(train_df.info())

print("\n--- Missing Values ---")
display(train_df.isnull().sum())

## Exploratory Data Analysis (EDA)\nLet's visualize the data to understand the underlying patterns and relationships, especially regarding the survival rate.

In [ ]:
# 1. Survival rate overview
plt.figure(figsize=(6, 4))
sns.countplot(x='Survived', data=train_df, palette='Set2')
plt.title('Survival Count (0 = No, 1 = Yes)')
plt.show()

survival_rate = train_df['Survived'].mean() * 100
print(f"Overall Survival Rate: {survival_rate:.2f}%")

In [ ]:
# 2. Survival by Sex and Pclass
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x='Sex', hue='Survived', data=train_df, palette='Set1', ax=axes[0])
axes[0].set_title('Survival by Sex')

sns.countplot(x='Pclass', hue='Survived', data=train_df, palette='Set3', ax=axes[1])
axes[1].set_title('Survival by Passenger Class')

plt.tight_layout()
plt.show()

In [ ]:
# 3. Age distribution by survival
plt.figure(figsize=(8, 5))
sns.histplot(data=train_df, x='Age', hue='Survived', multiple='stack', bins=30, palette='coolwarm')
plt.title('Age Distribution by Survival')
plt.show()

In [ ]:
# 4. Correlation analysis & Missing value analysis
# Missing values heatmap
plt.figure(figsize=(10, 4))
sns.heatmap(train_df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

# Correlation heatmap for numeric features
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(8, 6))
sns.heatmap(train_df[numeric_cols].corr(), annot=True, cmap='RdBu_r', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

## Feature Engineering\nApply custom feature engineering steps using the `TitanicFeatureEngineer` class from our `src` package to create new informative variables.

In [ ]:
# Apply feature engineering
feature_engineer = TitanicFeatureEngineer()

# Fit on training data and transform both
train_df_fe = feature_engineer.fit_transform(train_df)
test_df_fe = feature_engineer.transform(test_df)

print("Training data shape after feature engineering:", train_df_fe.shape)
print("Testing data shape after feature engineering:", test_df_fe.shape)

In [ ]:
# Display the engineered features
print("New columns added during feature engineering:")
new_cols = set(train_df_fe.columns) - set(train_df.columns)
print(new_cols)

display(train_df_fe.head())

## Data Preprocessing\nPrepare the data for modeling by applying scaling, imputation, and encoding through the preprocessor pipeline.

In [ ]:
# Separate features and target
X_train = train_df_fe.drop(config.TARGET_COL, axis=1)
y_train = train_df_fe[config.TARGET_COL]
X_test = test_df_fe.copy()

# Get the preprocessor
preprocessor = get_preprocessor(config)

# Fit and transform
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

print("Preprocessed X_train shape:", X_train_preprocessed.shape)
print("Preprocessed X_test shape:", X_test_preprocessed.shape)

## Model Training\nEvaluate individual base models using cross-validation, and then train a Stacking Ensemble for better predictive performance.

In [ ]:
# 1. Train individual models with CV
base_models = get_base_models(config)
print("Evaluating base models with Cross-Validation...")
cv_results = evaluate_individual_models(base_models, X_train_preprocessed, y_train, config)

# Display results
results_df = pd.DataFrame(cv_results).T
display(results_df.sort_values(by='mean_test_score', ascending=False))

In [ ]:
# 2. Train the Stacking Ensemble
print("Training Stacking Ensemble...")
ensemble = StackingEnsemble(base_models, config)
ensemble.fit(X_train_preprocessed, y_train)
print("Stacking Ensemble training completed.")

## Evaluation\nReview the final evaluation report comparing model performances.

In [ ]:
# Print formatted evaluation report
print_evaluation_report(cv_results)

# Alternative: end-to-end pipeline run
# training_result = train_pipeline(config)
# print(f"Pipeline ensemble score: {training_result.ensemble_score}")

## Generate Submission\nGenerate the final submission CSV file for Kaggle and validate its format.

In [ ]:
# Generate predictions using the trained ensemble
test_predictions = ensemble.predict(X_test_preprocessed)

# Define submission path (Kaggle requires it in /kaggle/working/)
out_dir = '/kaggle/working' if IS_KAGGLE else config.PROCESSED_DATA_DIR if hasattr(config, 'PROCESSED_DATA_DIR') else '.'
submission_path = os.path.join(out_dir, 'submission.csv')

# Generate and validate
generate_submission(test_df, test_predictions, submission_path)
validate_submission(submission_path)

print(f"Submission successfully saved to: {submission_path}")